In [6]:
import sys
sys.path.append("/home/onyxia/work/skill_match")

In [7]:
from skill_match.utils import read_uploaded_file

## NER

In [8]:
resume_path = "/home/onyxia/work/skill_match/notebooks/resume.txt"
jd_path = "/home/onyxia/work/skill_match/notebooks/job_description.txt"

with open(resume_path, "rb") as f:
    resume_txt = read_uploaded_file(f)

with open(jd_path, "rb") as f:
    jd_txt = read_uploaded_file(f)


In [18]:
import importlib
from skill_match.predict import match_cv_jd
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from skill_match.predict import extract_skills

In [19]:
_ner_model_name = "Nucha/Nucha_SkillNER_BERT"
_ner_model = AutoModelForTokenClassification.from_pretrained(_ner_model_name)
_ner_tokenizer = AutoTokenizer.from_pretrained(_ner_model_name)
_ner_pipeline = pipeline("ner", model=_ner_model, tokenizer=_ner_tokenizer, aggregation_strategy="simple")
_sentence_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3671.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
print("Extraction des compétences du CV...")
cv_skills = extract_skills(resume_txt, _ner_pipeline)

print("Extraction des compétences de la JD...")
jd_skills = extract_skills(jd_txt, _ner_pipeline)


Extraction des compétences du CV...
Extraction des compétences de la JD...


In [23]:
cv_skills

['with',
 'data analysis',
 'data',
 'machine learning',
 'english',
 's degree',
 'master']

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4790.46it/s]
BertForTokenClassification LOAD REPORT from: jjzha/jobbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can

[{'entity_group': 'LABEL_1', 'score': np.float32(0.5986747), 'word': 'Python SQL', 'start': 0, 'end': 10}, {'entity_group': 'LABEL_0', 'score': np.float32(0.5749514), 'word': 'machine', 'start': 11, 'end': 18}, {'entity_group': 'LABEL_1', 'score': np.float32(0.56787324), 'word': 'learning data science', 'start': 19, 'end': 40}]


In [24]:
jd_skills

['big data',
 'innovation',
 'e - commerce',
 '##ive',
 'python',
 'sql',
 'machine learning',
 'problem',
 'professional development',
 'solving',
 'data visualization',
 'communication',
 'motivated',
 's degree',
 'master',
 'azure',
 'data science']

In [ ]:
# AIDE AU NER

In [1]:
import requests
import csv
import shutil
import os

def download_esco_skills(cache_path="/home/onyxia/work/skill_match/data/esco_skills.csv", language="en"):
    print("Téléchargement des compétences ESCO...")
    skills = []
    
    next_url = f"https://ec.europa.eu/esco/api/search?type=skill&language={language}&limit=100&offset=0&full=true"

    while next_url:
        response = requests.get(next_url)
        if response.status_code != 200:
            break
        data = response.json()
        results = data.get("_embedded", {}).get("results", [])
        if not results:
            break

        for r in results:
            skill_types = r.get("hasSkillType", [])
            if any("attitude" in st for st in skill_types):
                continue
            skills.append(r["title"].lower())
        
        next_url = data.get("_links", {}).get("next", {}).get("href")
        print(f"{len(skills)} compétences récupérées", end="\r")

    with open(cache_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["skill"])
        for skill in sorted(set(skills)):
            writer.writerow([skill])
    
    print(f"\nDone ! {len(skills)} compétences sauvegardées")

# il faut lancer ça une fois stp Boubaker 
download_esco_skills()

Téléchargement des compétences ESCO...
1700 compétences récupérées
Done ! 1700 compétences sauvegardées


In [3]:
import kagglehub
import shutil
import os

# Télécharger le dataset
path = kagglehub.dataset_download("ravindrasinghrana/job-description-dataset")

print("Dataset téléchargé ici :", path)

# Dossier de destination
dest = "/home/onyxia/work/skill_match/data/"

# Copier les fichiers dans ton dossier de travail
for file in os.listdir(path):
    src_file = os.path.join(path, file)
    dest_file = os.path.join(dest, file)
    shutil.copy(src_file, dest_file)
    print(f"Copié : {file}")

print("Tous les fichiers ont été copiés ✅")

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset téléchargé ici : /home/onyxia/.cache/kagglehub/datasets/ravindrasinghrana/job-description-dataset/versions/1
Copié : job_descriptions.csv
Tous les fichiers ont été copiés ✅


In [4]:
import pandas as pd
import re

df = pd.read_csv("/home/onyxia/work/skill_match/data/job_descriptions.csv")

all_skills = set()

for row in df["skills"].dropna():
    for skill in row.split("\n"):
        skill = skill.strip().lower()
        if not skill:
            continue
        
        # contenu entre parenthèses
        parens = re.findall(r'\(e\.g\.,?\s*([^)]+)\)', skill)
        for p in parens:
            for item in p.split(","):
                all_skills.add(item.strip())

        clean = re.sub(r'\(.*?\)', '', skill).strip()
        if clean:
            all_skills.add(clean)

print(f"{len(all_skills)} compétences extraites")

493 compétences extraites


In [ ]:
# fusion de esco et skills du dataset job_description

esco_df = pd.read_csv("/home/onyxia/work/skill_match/data/esco_skills.csv")
esco_skills = set(esco_df["skill"].str.lower())

combined = esco_skills | all_skills

print(f"{len(combined)} compétences au total")

combined_df = pd.DataFrame(sorted(combined), columns=["skill"])
combined_df.to_csv("/home/onyxia/work/skill_match/data/esco_skills.csv", index=False)

print("Sauvegardé !")

2191 compétences au total
Sauvegardé !
'power bi' présent : True
